In [1]:
import pandas as pd
from pathlib import Path

In [3]:
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "raw"
REPORTS_DIR = BASE_DIR / "reports"

REPORTS_DIR.mkdir(exist_ok=True)

In [4]:
# 2. Load data

spending = pd.read_csv(DATA_DIR / "spending_profile.csv")
cards = pd.read_csv(DATA_DIR / "credit_cards.csv")

In [5]:
# 3. Calculate annual spending by category

spending["annual_spending"] = (
    spending["monthly_spending"] * 12
)

annual_spending = dict(
    zip(
        spending["category"],
        spending["annual_spending"]
    )
)

total_annual_spending = spending["annual_spending"].sum()


In [6]:
# 4. Reward rate mapping

rate_columns = {
    "Groceries": "groceries_rate",
    "Dining": "dining_rate",
    "Gas": "gas_rate",
    "Travel": "travel_rate",
    "Bills": "bills_rate",
    "Other": "other_rate"
}

In [7]:
# 5. Calculate rewards and welcome bonus eligibility

results = []

for _, card in cards.iterrows():

    annual_rewards = 0

    for category, annual_amount in annual_spending.items():

        rate_column = rate_columns[category]

        reward_rate = card[rate_column]

        category_rewards = annual_amount * reward_rate

        annual_rewards += category_rewards

    annual_fee = card["annual_fee"]

   
    # Estimate spending during the welcome bonus period

    bonus_months = card["welcome_bonus_months"]

    minimum_spending = card["welcome_bonus_min_spend"]

    estimated_bonus_period_spending = (
        total_annual_spending * bonus_months / 12
    )

   
    # Check bonus eligibility
    
    bonus_eligible = (
        estimated_bonus_period_spending >= minimum_spending
    )

    if bonus_eligible:
        welcome_bonus = card["welcome_bonus_value"]
    else:
        welcome_bonus = 0

    # Calculate financial value
   
    ongoing_net_value = annual_rewards - annual_fee

    first_year_net_value = (
        annual_rewards
        - annual_fee
        + welcome_bonus
    )

    # Store results

    results.append({
        "card_name": card["card_name"],
        "annual_spending": total_annual_spending,
        "annual_rewards": annual_rewards,
        "annual_fee": annual_fee,
        "ongoing_net_value": ongoing_net_value,
        "estimated_bonus_period_spending":
            estimated_bonus_period_spending,
        "welcome_bonus_min_spend": minimum_spending,
        "welcome_bonus_eligible": bonus_eligible,
        "welcome_bonus_value": welcome_bonus,
        "first_year_net_value": first_year_net_value
    })


In [8]:
# 6. Create results DataFrame

results_df = pd.DataFrame(results)


In [9]:
# 7. Rank cards by ongoing value

results_df["ongoing_rank"] = (
    results_df["ongoing_net_value"]
    .rank(method="min", ascending=False)
)

In [10]:
# 8. Rank cards by first-year value

results_df["first_year_rank"] = (
    results_df["first_year_net_value"]
    .rank(method="min", ascending=False)
)

results_df = results_df.sort_values(
    "first_year_rank"
)

In [11]:
# 9. Export results

results_df.to_csv(
    REPORTS_DIR / "advanced_rewards_analysis.csv",
    index=False
)

In [12]:
# 10. Display results

print("\n" + "=" * 70)
print("ADVANCED CREDIT CARD REWARDS ANALYSIS")
print("=" * 70)

print(
    results_df[
        [
            "card_name",
            "annual_rewards",
            "annual_fee",
            "ongoing_net_value",
            "welcome_bonus_eligible",
            "welcome_bonus_value",
            "first_year_net_value",
            "ongoing_rank",
            "first_year_rank"
        ]
    ].to_string(index=False)
)


ADVANCED CREDIT CARD REWARDS ANALYSIS
              card_name  annual_rewards  annual_fee  ongoing_net_value  welcome_bonus_eligible  welcome_bonus_value  first_year_net_value  ongoing_rank  first_year_rank
      Capital One Savor           435.6           0              435.6                    True                  200                 635.6           2.0              1.0
Chase Freedom Unlimited           383.4           0              383.4                    True                  200                 583.4           3.0              2.0
Wells Fargo Active Cash           439.2           0              439.2                    True                  100                 539.2           1.0              3.0


In [13]:
# 11. Identify winners

best_ongoing = results_df.loc[
    results_df["ongoing_net_value"].idxmax()
]

best_first_year = results_df.loc[
    results_df["first_year_net_value"].idxmax()
]

print("\nBest card for ongoing value:")
print(best_ongoing["card_name"])

print(
    "Ongoing net value: $"
    f"{best_ongoing['ongoing_net_value']:,.2f}"
)

print("\nBest card for first-year value:")
print(best_first_year["card_name"])

print(
    "First-year net value: $"
    f"{best_first_year['first_year_net_value']:,.2f}"
)

print("\nReport generated:")
print("reports/advanced_rewards_analysis.csv")

print("\nAnalysis completed successfully!")


Best card for ongoing value:
Wells Fargo Active Cash
Ongoing net value: $439.20

Best card for first-year value:
Capital One Savor
First-year net value: $635.60

Report generated:
reports/advanced_rewards_analysis.csv

Analysis completed successfully!
